In [1]:

# Notebook 04: Healthcare Analytics Dashboard
# Step 9.3: Environment Setup

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

from plotly.subplots import make_subplots
from IPython.display import display, HTML
from google.colab import files

import zipfile
import os
import json
import joblib

print("Dashboard environment ready!")

uploaded = files.upload()


Dashboard environment ready!


Saving cleaned_data.zip to cleaned_data.zip


In [2]:
from google.colab import files
files.upload()

Saving ml_outputs (1).zip to ml_outputs (1).zip


{'ml_outputs (1).zip': b'PK\x03\x04\x14\x00\x00\x00\x08\x00\xd2Xy]\xa8\x08\xce-\xc1\x00\x00\x00\x14\x01\x00\x00\x14\x00\x00\x00model_comparison.csvM\x8eAj\x031\x0cE\xf7s\x8a9\x80\t\x96\xf5-\xc9\xcb\x10\xc8\xaa\xa1eJ\x0f08&\x0cLc\x98I\x17\xbd}\xed@K7\x02==\xf4\xff\xa5^\xcb\xea\x8e9\x7fms\xfevo[\xc9\xcb\xbe\xd4\xbb\x9bJ\x9e\xd7\xd5\x9di|\xcfu+nz=\x8d\xc7\x8f\xd3\xf0Ro\xcb\xfeX\xf28\x95\xdbV\xf6\xa7\xeb\x0f\xa2\xf0\xcc\x84DJA\xbd5\xc4l\x10\x8a\xbf36\x14\xd5\x12\x94\xc5\x10\xc8\xc7\xa0\r!(\x19 \xe2)\x81\x14]S\xa2 \xa19j\xa6\xac2L\xf3\xfdZ?\xc7s\xeb\xb1?\x9e\x82\x05\x0e\x1e\xa0d\x96"\xf54\x10\xe4\xdf\xe8\xbf\xff\x82\x84\xa9Gw\x14\x01\xf6\xacI!\xd6\x97\xde=Z\x8bkwk\x05Xi\xf8\x01PK\x03\x04\x14\x00\x00\x00\x08\x00\xd2Xy]\xf6\x1e;\xd7\xc1\x00\x00\x00\x10\x01\x00\x00\x1b\x00\x00\x00cross_validation_report.csv-\x8e\xcbj\xc3@\x0cE\xf7\xfe\n\x7f@(\x1a\x8d\x9e\xcb\xd4\x90UC!\x90\xec\xcdd\x08\x06\xd7\x03q\xba\xe8\xdfG6]\x08\xae\x0e\xd2\xe1\x9e\xdb\xbd\xce\x87\xe1\xd6\x1fK\xf9}\x8e\xe5o\xcb\x97Z\xc6y\

In [7]:
import os

if os.path.exists("ml_outputs (1).zip"):
    os.rename("ml_outputs (1).zip", "ml_outputs.zip")

print("Cleaned data:", os.path.exists("cleaned_data.zip"))
print("ML outputs:", os.path.exists("ml_outputs.zip"))

Cleaned data: True
ML outputs: True


In [8]:

# Step 9.4: Extract and Load Dashboard Data

for folder in ["cleaned_data", "eda_outputs", "ml_outputs"]:

    os.makedirs(f"/content/{folder}", exist_ok=True)

    with zipfile.ZipFile(f"/content/{folder}.zip", "r") as z:
        z.extractall(f"/content/{folder}")

df = pd.read_csv("/content/cleaned_data/master_dataset.csv")

print("DASHBOARD DATA REPORT")
print("=" * 45)

print("Total appointments:", len(df))
print("Total columns:", df.shape[1])
print("Total clinics:", df["clinic_id"].nunique())
print("Total cities:", df["city"].nunique())

print("\nDashboard data loaded successfully!")


DASHBOARD DATA REPORT
Total appointments: 1000
Total columns: 24
Total clinics: 8
Total cities: 6

Dashboard data loaded successfully!


In [5]:
from google.colab import files
uploaded = files.upload()

Saving eda_outputs.zip to eda_outputs.zip


In [9]:

# Step 9.5: Executive Dashboard KPI Cards

total = len(df)

attended = (df["status"] == "Attended").sum()
no_show = (df["status"] == "No-Show").sum()
known = attended + no_show

no_show_rate = no_show / known * 100

revenue_risk = df.loc[
    df["status"] == "No-Show", "fee_inr"
].sum()

fig = make_subplots(
    rows=1,
    cols=4,
    specs=[[{"type": "indicator"}] * 4]
)

metrics = [
    ("Total Appointments", total, "", ""),
    ("Attended", attended, "", ""),
    ("No-Show Rate", no_show_rate, "", "%"),
    ("Revenue at Risk", revenue_risk, "₹", "")
]

for i, (label, value, prefix, suffix) in enumerate(metrics):

    fig.add_trace(
        go.Indicator(
            mode="number",
            value=value,
            title={"text": label, "font": {"size": 17}},
            number={
                "prefix": prefix,
                "suffix": suffix,
                "valueformat": ".2f" if suffix == "%" else ",.0f",
                "font": {
                    "size": 36,
                    "color": "#D97706" if i >= 2 else "#0B6B4F"
                }
            }
        ),
        row=1,
        col=i+1
    )

fig.update_layout(
    title={
        "text": "HEALTHCARE PERFORMANCE DASHBOARD",
        "x": 0.5
    },
    paper_bgcolor="#F5F8F7",
    height=300,
    margin=dict(t=100, b=30, l=20, r=20)
)

os.makedirs("/content/dashboard", exist_ok=True)

fig.write_html("/content/dashboard/kpi_dashboard.html")

fig.show()

print("KPI dashboard created successfully!")


KPI dashboard created successfully!


In [10]:

# Step 9.6: Interactive Clinic Performance

valid_df = df[df["status"].isin(
    ["Attended", "No-Show"]
)].copy()

valid_df["no_show_flag"] = (
    valid_df["status"] == "No-Show"
).astype(int)

clinic_summary = valid_df.groupby(
    "clinic_name"
).agg(
    Total_Appointments=("status", "count"),
    No_Shows=("no_show_flag", "sum")
).reset_index()

clinic_summary["No_Show_Rate"] = (
    clinic_summary["No_Shows"] /
    clinic_summary["Total_Appointments"] * 100
).round(2)

clinic_summary = clinic_summary.sort_values(
    "No_Show_Rate"
)

fig_clinic = px.bar(
    clinic_summary,
    x="No_Show_Rate",
    y="clinic_name",
    orientation="h",
    color="No_Show_Rate",
    color_continuous_scale=[
        "#0B6B4F", "#D4A73C", "#DC5A45"
    ],
    text="No_Show_Rate",
    hover_data=[
        "Total_Appointments", "No_Shows"
    ],
    title="Clinic Performance | No-Show Risk"
)

fig_clinic.update_traces(
    texttemplate="%{text:.2f}%",
    textposition="outside"
)

fig_clinic.add_vline(
    x=no_show_rate,
    line_dash="dash",
    line_color="red",
    annotation_text="Overall Average"
)

fig_clinic.update_layout(
    template="plotly_white",
    height=550,
    xaxis_title="No-Show Rate (%)",
    yaxis_title="Clinic",
    coloraxis_showscale=False,
    xaxis_range=[
        0, clinic_summary["No_Show_Rate"].max()+10
    ]
)

fig_clinic.write_html(
    "/content/dashboard/clinic_dashboard.html"
)

fig_clinic.show()

print("Clinic dashboard created successfully!")


Clinic dashboard created successfully!


In [11]:

# Step 9.7: Reminder Effectiveness Dashboard

reminder_summary = valid_df.groupby(
    "reminder_sent"
).agg(
    Total_Appointments=("status", "count"),
    No_Shows=("no_show_flag", "sum")
).reset_index()

reminder_summary["No_Show_Rate"] = (
    reminder_summary["No_Shows"] /
    reminder_summary["Total_Appointments"] * 100
).round(2)

fig_reminder = px.bar(
    reminder_summary,
    x="reminder_sent",
    y="No_Show_Rate",
    color="reminder_sent",
    color_discrete_map={
        "Yes": "#0B6B4F",
        "No": "#DC5A45"
    },
    text="No_Show_Rate",
    hover_data=[
        "Total_Appointments",
        "No_Shows"
    ],
    title="Appointment Reminders and No-Show Risk"
)

fig_reminder.update_traces(
    texttemplate="%{text:.2f}%",
    textposition="outside",
    width=0.45
)

fig_reminder.update_layout(
    template="plotly_white",
    height=450,
    showlegend=False,
    xaxis_title="Reminder Sent",
    yaxis_title="No-Show Rate (%)",
    yaxis_range=[0, 40]
)

fig_reminder.write_html(
    "/content/dashboard/reminder_dashboard.html"
)

fig_reminder.show()

print("Reminder dashboard created successfully!")


Reminder dashboard created successfully!


In [12]:

# Step 9.8: Combined Executive Dashboard

dashboard = make_subplots(
    rows=3,
    cols=4,
    specs=[
        [{"type": "indicator"}] * 4,
        [
            {"type": "pie", "colspan": 2},
            None,
            {"type": "xy", "colspan": 2},
            None
        ],
        [
            {"type": "xy", "colspan": 4},
            None, None, None
        ]
    ],
    subplot_titles=[
        "", "", "", "",
        "Appointment Outcomes",
        "Reminder Analysis",
        "Clinic-wise No-Show Performance"
    ],
    row_heights=[0.20, 0.35, 0.45],
    vertical_spacing=0.10
)

# KPI Cards

for i, (label, value, prefix, suffix) in enumerate(metrics):

    dashboard.add_trace(
        go.Indicator(
            mode="number",
            value=value,
            title={"text": label},
            number={
                "prefix": prefix,
                "suffix": suffix,
                "valueformat": ".2f" if suffix == "%" else ",.0f",
                "font": {
                    "size": 32,
                    "color": "#0B6B4F" if i < 2 else "#D97706"
                }
            }
        ),
        row=1, col=i+1
    )

# Appointment Outcomes

dashboard.add_trace(
    go.Pie(
        labels=["Attended", "No-Show", "Unknown"],
        values=[attended, no_show, total-known],
        hole=0.65,
        marker=dict(
            colors=["#0B6B4F", "#DC5A45", "#CBD5E1"]
        ),
        textinfo="percent+label"
    ),
    row=2, col=1
)

# Reminder Analysis

dashboard.add_trace(
    go.Bar(
        x=reminder_summary["reminder_sent"],
        y=reminder_summary["No_Show_Rate"],
        marker_color=[
            "#0B6B4F" if x == "Yes" else "#DC5A45"
            for x in reminder_summary["reminder_sent"]
        ],
        text=reminder_summary["No_Show_Rate"],
        texttemplate="%{text:.2f}%",
        textposition="outside"
    ),
    row=2, col=3
)

# Clinic Performance

dashboard.add_trace(
    go.Bar(
        x=clinic_summary["No_Show_Rate"],
        y=clinic_summary["clinic_name"],
        orientation="h",
        marker_color=[
            "#DC5A45" if x > no_show_rate else "#0B6B4F"
            for x in clinic_summary["No_Show_Rate"]
        ],
        text=clinic_summary["No_Show_Rate"],
        texttemplate="%{text:.2f}%",
        textposition="outside"
    ),
    row=3, col=1
)


dashboard.add_shape(
    type="line",
    x0=no_show_rate,
    x1=no_show_rate,
    y0=0,
    y1=1,
    xref="x2",
    yref="y2 domain",
    line=dict(
        color="#64748B",
        width=2,
        dash="dash"
    )
)

dashboard.update_yaxes(
    title_text="No-Show Rate (%)",
    range=[0, 40],
    row=2, col=3
)

dashboard.update_xaxes(
    title_text="No-Show Rate (%)",
    range=[0, clinic_summary["No_Show_Rate"].max()+10],
    row=3, col=1
)

dashboard.update_layout(
    title={
        "text": "<b>HEALTHCARE INTELLIGENCE DASHBOARD</b>",
        "x": 0.5
    },
    template="plotly_white",
    paper_bgcolor="#F5F8F7",
    height=1050,
    showlegend=False,
    margin=dict(l=180, r=60, t=100, b=60)
)

dashboard.write_html(
    "/content/dashboard/executive_dashboard.html"
)

dashboard.show()

print("Executive dashboard created successfully!")


Executive dashboard created successfully!


In [13]:

# Step 9.9: Export Dashboard Files

import shutil

print("DASHBOARD EXPORT REPORT")
print("=" * 45)

for filename in sorted(os.listdir("/content/dashboard")):
    print(filename)

shutil.make_archive(
    "/content/dashboard_outputs",
    "zip",
    "/content/dashboard"
)

print("\nAll dashboards exported successfully!")
print("Notebook 04 completed successfully!")

files.download("/content/dashboard_outputs.zip")


DASHBOARD EXPORT REPORT
clinic_dashboard.html
executive_dashboard.html
kpi_dashboard.html
reminder_dashboard.html

All dashboards exported successfully!
Notebook 04 completed successfully!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>